# Native classical shadow tomography

Prepare the two-bit Bell state $|\Phi^-\rangle$, collect **400 local Pauli snapshots** and **50 depth-20 random Clifford snapshots**, and reconstruct both shadow estimates. Each snapshot is one actual seeded draw from a native FlagQuantum quantum execution. The source's random seed 552, basis angles and H/S/CX random walk are preserved.

No Qmod source, importer, remote service or credentials are needed. [WORKFLOW.md](WORKFLOW.md) maps the pinned source workflow and explains local sampling adaptations.

## Declare the quantum measurement experiment

The editable Pauli probe below prepares a Bell state and composes the selected
basis rotations. Basis choices are classical bindings; the q array is quantum.
FlagQuantum builds each bound circuit and returns named logical probabilities.
The complete driver still executes all 450 original single-shot snapshots,
including the separate random Clifford ensemble, before classical reconstruction.

Try changing one probe basis and inspect how correlations change. Each snapshot
is an actual sampled outcome, not its exact expected observable. More snapshots
can reduce sampling error without making a finite-depth Clifford walk an exact
design. A reconstructed matrix with a negative eigenvalue is possible for linear
inversion; do not quietly turn it into a physical state by clipping eigenvalues.


In [ ]:
from matplotlib import pyplot as plt

from examples.synthesis.algorithms.metrology.classical_shadow_tomography.overview.workflow import (
    plot_workflow,
    run_workflow,
)

print("Local seed: 552; data bits: 2; one shot per snapshot")
print("Pauli: 400 snapshots; Clifford: 50 snapshots of depth 20")

import math
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QArray, QBit, R, RZ, prepare_bell_state, qfunc


@qfunc
def pauli_probe(
    thetas: list[float], phis: list[float], q: Output[QArray[QBit]]
) -> None:
    prepare_bell_state(1, q)
    for i in range(2):
        RZ(phis[i], q[i])
        R(thetas[i], math.pi / 2, q[i])


preview = synth.synthesize(
    pauli_probe,
    classical_inputs={"thetas": [0.0, 0.0], "phis": [0.0, 0.0]},
    max_qubits=2,
    max_workspace=0,
    memory_limit_bytes=1024**2,
)
print(
    "Unrotated Bell probabilities:",
    dict(preview.probabilities(fq.run(preview), registers=("q",))),
)


## Pauli shadow: 400 single shots

Each bit independently chooses X, Y or Z. Native qfunc prepares the Bell state and applies RZ followed by RY (expressed as `R(theta, pi/2)`). Static parameter binding creates the actual local circuit for each snapshot; this does not claim the source's remote single-template execution interface.

The inverse local channel is $3U^\dagger|b\rangle\langle b|U-I$. For Y, the HS basis change maps $|+y\rangle$ to outcome **1**, so its sign must be reversed. Logical index zero is the first matrix tensor factor; it is the low bit of the native named integer readout.

In [ ]:
workflow = run_workflow()
assert workflow.pauli.bases is not None
print("First ten Pauli snapshots:", workflow.pauli.bits[:10].tolist())
print("First ten basis choices:", workflow.pauli.bases[:10].tolist())
print("Pauli reconstruction:", workflow.pauli.density)
print("Pauli Frobenius error:", workflow.pauli.frobenius_error)
print("Median of two means, ZZ:", workflow.pauli.zz_estimate)


## Global Clifford shadow: 50 single shots

Preserve each sampled 20-gate H/S/CX sequence and reconstruct $(2^2+1)U^\dagger|b\rangle\langle b|U-I$. Report the conventional median of ten block means for ZZ; for an even number of blocks, average the middle two means.

A finite-depth random Clifford walk is an **approximate ensemble**. It is not certified here as an exact uniform Clifford design. Linear-inversion estimates may have negative eigenvalues; they are not silently clipped or normalized into physical density matrices.

In [ ]:
print("First five Clifford snapshots:", workflow.clifford.bits[:5].tolist())
print("First recorded gate sequence:", workflow.clifford.sequences[0])
print("Clifford reconstruction:", workflow.clifford.density)
print("Clifford Frobenius error:", workflow.clifford.frobenius_error)
print("Median of ten means, ZZ:", workflow.clifford.zz_estimate)


## Actual observations and theoretical budgets

Compute the original single-observable error formulas at error 0.2 and failure probability 0.01. The results are **81065** Pauli samples and **108086** ideal-Clifford samples, with ten divisions. Only the original 400/50 snapshots are executed. Neither finite observation claims the theoretical error guarantee, and the ideal-Clifford bound does not certify the finite-depth walk.

NumPy RandomState reproduces the source basis/gate/seed stream; local Torch sampling explicitly replaces the remote sampler. Identical original remote shot outcomes are not claimed. This is a local `single_device_fast_path`, not distributed or hardware evidence.

In [ ]:
print("Computed Pauli sample bound / divisions:", workflow.pauli_bound)
print("Computed ideal-Clifford sample bound / divisions:", workflow.clifford_bound)
print("These budgets were computed, not executed.")
for name, shadow in (("Pauli", workflow.pauli), ("Clifford", workflow.clifford)):
    print(name, "actual single shots:", len(shadow.bits))
    print(
        "Per-circuit operation range:",
        min(r.n_operations for r in shadow.resources),
        max(r.n_operations for r in shadow.resources),
    )
    print(
        "Per-circuit depth range:",
        min(r.depth for r in shadow.resources),
        max(r.depth for r in shadow.resources),
    )
print("Each bound circuit has two physical bits and zero clean work bits.")


## Measured reconstructions

The plots show actual linear-inversion matrices and measured ZZ estimates. Color limits cover the observed entries, including negative entries. The true Bell-state ZZ value is one; finite sampled reconstruction errors are reported without a convergence or quantum-advantage claim.

In [ ]:
plt.show()
figure = plot_workflow(workflow)
figure
